In [1]:
import numpy as np
import pandas as pd
import time
import json
import joblib
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

import sys
sys.path.append('../src') # Чтобы ноутбук увидел папку src
from config import METHODS_DICT, DATASETS, RESULTS_PATH, get_paths
from visualizer import get_pca_plot, get_tsne_plot, compute_tsne_sample, get_tsne_embedding
from data_loader import get_target_names
from vectors_loader import load_vectors

In [13]:
# DATASET_NAME = "20 Newsgroups"
# DATASET_NAME = "AG News"
DATASET_NAME = "IMDB"
# DATASET_NAME = "Udmurt media"
dataset_key = DATASETS[DATASET_NAME]["key"]
dataset_n_classes = DATASETS[DATASET_NAME]["n_classes"]

paths = get_paths(dataset_key)

VECTORS_PATH = paths["VECTORS"]
PLOTS_PATH = paths["PLOTS"]
TABLES_PATH = paths["TABLES"]
PRED_PATH = paths["PRED"]
MODELS_PATH = paths["MODELS"]
print(f"📊 Работаем с датасетом: {DATASET_NAME}")
print("Папки с которыми работаем")
print(VECTORS_PATH)
print(PLOTS_PATH)
print(TABLES_PATH)
print(PRED_PATH)
print(MODELS_PATH)

os.makedirs(PLOTS_PATH, exist_ok=True)
os.makedirs(TABLES_PATH, exist_ok=True)
os.makedirs(PRED_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

📊 Работаем с датасетом: IMDB
Папки с которыми работаем
D:\PythonProj\Diplom_v2\data\processed/imdb
D:\PythonProj\Diplom_v2\results\plots/imdb
D:\PythonProj\Diplom_v2\results\tables/imdb
D:\PythonProj\Diplom_v2\results\predictions/imdb
D:\PythonProj\Diplom_v2\data\models/imdb


In [14]:
# 1. Загрузка меток и временных статистик
y_train = np.load(f'{VECTORS_PATH}/y_train.npy')
y_test = np.load(f'{VECTORS_PATH}/y_test.npy')
y_all = np.concatenate([y_train, y_test])

with open(f'{VECTORS_PATH}/vectorization_time.json', 'r') as f:
    v_times = json.load(f)


In [34]:
results = []
for name, key in METHODS_DICT.items():
    print(f"\n{name}")
    
    # 1. Загрузка векторов
    # Объединяем train и test, чтобы кластеризовать весь датасет
    X_train, X_test = load_vectors(VECTORS_PATH, key, dense=True)
    X_all = np.vstack([X_train, X_test])
    
    start_time = time.time()
    
    # 2. Предобработка (Нормализация обязательна для KMeans)
    X_norm = normalize(X_all)

    # 3. Снижение размерности (PCA) 
    # Для разреженных методов (TF-IDF) это критично, для BERT опционально, но оставим для честности сравнения
    pca = PCA(n_components=50, random_state=42)
    X_reduced = pca.fit_transform(X_norm)
    explained_var = sum(pca.explained_variance_ratio_)
    print(f"Explained variance (50 components): {explained_var:.4f}")
    
    # 4. Кластеризация KMeans
    kmeans = KMeans(n_clusters=dataset_n_classes, random_state=42, n_init=10, init="k-means++")
    clusters = kmeans.fit_predict(X_reduced)

    # сохраняем кластеры
    np.save(f"{PRED_PATH}/{key}_clusters.npy", clusters)

    # сохраняем модель (необязательно но пусть будет)
    joblib.dump(kmeans, f"{MODELS_PATH}/{key}_kmeans.pkl")

    # 5. Расчет метрик
    # Silhouette считаем на уменьшенных данных для скорости
    # насколько хорошо точки внутри кластера похожи
    # и насколько далеко от других кластеров
    sil = silhouette_score(X_reduced, clusters, sample_size=6000, random_state=42)
    # насколько кластеры совпали с реальными классами
    ari = adjusted_rand_score(y_all, clusters)
    
    duration = time.time() - start_time
    total_time = v_times[key] + duration
    
    # print(f"Silhouette: {sil:.4f}")
    # print(f"ARI: {ari:.4f}")
    # print(f"Model Time: {duration}s (Vec Time: {vec_time}s)")

    results.append( {
        "Method": name,
        "Silhouette": sil,
        "ARI": ari,
        "Vectorization Time (s)": round(v_times[key],2),
        "Model Training Time (s)": round(duration,2),
        "Total Time (s)": round(total_time,2)
    })


Binary
Explained variance (50 components): 0.2961

Bag of Words
Explained variance (50 components): 0.3812

TF-IDF Standard
Explained variance (50 components): 0.2473

TF-IDF Bigrams
Explained variance (50 components): 0.2687

Word2Vec
Explained variance (50 components): 0.9999

Doc2Vec
Explained variance (50 components): 0.9071

BERT
Explained variance (50 components): 0.8650


In [35]:
# 6. Итоговая таблица
results_df = pd.DataFrame(results).sort_values(by="ARI", ascending=False)

print("\nИтоговая сравнительная таблица кластеризации")
print(results_df)

# Сохраняем расширенные результаты
results_df.to_csv(f"{TABLES_PATH}/clusterization_compare.csv", index=False)


Итоговая сравнительная таблица кластеризации
            Method  Silhouette       ARI  Vectorization Time (s)  \
5          Doc2Vec    0.038592  0.348013                   17.79   
2  TF-IDF Standard    0.130704  0.230022                    0.46   
4         Word2Vec    0.179921  0.205267                    2.30   
0           Binary    0.121180  0.177833                    0.35   
3   TF-IDF Bigrams    0.148343  0.158863                    1.06   
1     Bag of Words    0.124505  0.125034                    0.38   
6             BERT    0.067080  0.033233                  136.62   

   Model Training Time (s)  Total Time (s)  
5                     0.41           18.20  
2                     4.04            4.50  
4                     0.45            2.75  
0                     4.46            4.81  
3                     3.91            4.97  
1                     4.34            4.71  
6                     1.32          137.95  


In [36]:
# TO DO:
# Добавить еще расчет t-SNE (написать функцию в файл ui_helpers, visualizer.py (по аналогии с рса))
# Спросить у чата, как модно визуализировать результаты кластеризации (как посмотреть как модель разбила на кластеры все? и надо ли это ?)

In [8]:
# Выбираем топ два метода
results_df= pd.read_csv(f"{TABLES_PATH}/clusterization_compare.csv")
top_methods = results_df.head(2)["Method"].values
with open(f"{RESULTS_PATH}/{dataset_key}_top_methods_clusterization.json", "w") as f:
    json.dump(list(top_methods), f)
print("TOP METHODS:")
for m in top_methods:
    print(m)

TOP METHODS:
BERT
Doc2Vec


In [15]:
# 2. Запускаем отрисовку
target_names = get_target_names(dataset_key)
for name in top_methods:
    key = METHODS_DICT[name]
    print(f"\n>>> Визуализация метода: {name}")
    
    # Загружаем всё необходимое
    X_train, X_test = load_vectors(VECTORS_PATH, key, dense=True)
    X_all = np.vstack([X_train, X_test])
    X_norm = normalize(X_all)
    
    # Предсказанные кластеры (мы их сохраняли в цикле)
    clusters = np.load(f"{PRED_PATH}/{key}_clusters.npy")
    
    # Для визуализации нам всё равно нужно сжать данные до 50 компонент (как в цикле)
    pca = PCA(n_components=50, random_state=42)
    X_reduced = pca.fit_transform(X_norm)
    
    # --- РИСУЕМ PCA (2D) ---
    print("Отрисовка PCA...")
    # Реальные темы
    fig_pca_real = get_pca_plot(X_reduced, y_all, target_names, f"{name} (Real Topics)")
    fig_pca_real.savefig(f"{PLOTS_PATH}/clustering_{key}_pca_real.png", dpi=300, bbox_inches='tight')
    plt.close(fig_pca_real)
    
    # Предсказанные кластеры
    fig_pca_pred = get_pca_plot(X_reduced, clusters, target_names, f"{name} (KMeans Clusters)")
    fig_pca_pred.savefig(f"{PLOTS_PATH}/clustering_{key}_pca_pred.png", dpi=300, bbox_inches='tight')
    plt.close(fig_pca_pred)

    print("Отрисовка t-SNE ...")
    # --- РИСУЕМ t-SNE ---
    X_sample, idx = compute_tsne_sample(X_reduced)
    X_tsne = get_tsne_embedding(X_sample)
    # Реальные темы
    y_real = y_all[idx] if idx is not None else y_all
    fig_tsne_real = get_tsne_plot(X_tsne, y_real, target_names, name, label_type='real')
    fig_tsne_real.savefig(f"{PLOTS_PATH}/clustering_{key}_tsne_real.png", 
                         dpi=300, bbox_inches='tight')
    plt.close(fig_tsne_real)

    # Предсказанные кластеры
    y_pred = clusters[idx] if idx is not None else clusters
    fig_tsne_pred = get_tsne_plot(X_tsne, y_pred, target_names, name, label_type='pred')
    fig_tsne_pred.savefig(f"{PLOTS_PATH}/clustering_{key}_tsne_pred.png", 
                         dpi=300, bbox_inches='tight')
    plt.close(fig_tsne_pred)

print("\nВсе графики успешно сохранены!")


>>> Визуализация метода: BERT
Отрисовка PCA...
Отрисовка t-SNE (подвыборка 6000 точек)...
t-SNE: выбрано 10000 точек из 50000 (подвыборка)
Perplexity = 50

>>> Визуализация метода: Doc2Vec
Отрисовка PCA...
Отрисовка t-SNE (подвыборка 6000 точек)...
t-SNE: выбрано 10000 точек из 50000 (подвыборка)
Perplexity = 50

Все графики успешно сохранены!
